# Analytical verification of the pix11 dataset

End-to-end quantitative check on `data/visual/pix11/` — the bin where ImageGen actually produced distinct data (`dp=2`, ~10 px max displacement) for this sim run.

**Six layers of verification, weakest to strongest:**

1. **File structure & per-pair stats** — does the file contain the right number of pairs, with sensible flow magnitudes?
2. **Visual: frames A vs B** — particle blobs should appear in nearly the same places with small coherent shifts.
3. **Flow field anatomy** — u/v heatmaps, speed+quiver. Are the field statistics consistent across the bin?
4. **Displacement distribution** — histograms with percentiles. Is the data consistent with `pix=11` (target ~10 px max)?
5. **Patch cross-correlation** — measured shift between A and B patches vs predicted shift from the saved flow.
6. **Per-particle displacement (gold standard)** — bypass the rendering entirely. Use raw particle positions from the combined file. For every one of the 16,384 particles: compute its actual movement, look up the saved flow at its starting position via bilinear interpolation, compare. RMSE in pixels gives the final verdict.

If RMSE at step 6 is sub-pixel, the entire pipeline is internally consistent and the data is ready for ML training.

In [ ]:
from pathlib import Path
import numpy as np
import h5py
import matplotlib.pyplot as plt

PAIR = 1   # focus pair for detailed analysis (1..n_pairs)

# JLD2 sometimes stores arrays as object references; this peels them.
def deref(obj, h5):
    if isinstance(obj, h5py.Reference):
        return deref(h5[obj][()], h5)
    if isinstance(obj, np.ndarray):
        if obj.dtype == object:
            if obj.size == 1:
                return deref(obj.flat[0], h5)
            return np.array([deref(x, h5) for x in obj.flat]).reshape(obj.shape)
        return obj
    return obj

pix_path = next(Path('data/visual/pix11').glob('*image_pairs*.jld2'))
combined_path = next(Path('data/binary').glob('*_combined.jld2'))
print('pix11 file   :', pix_path.name)
print('combined file:', combined_path.name)

## 1. File structure & per-pair statistics

List every pair in the file, the max/mean speed each carries, and infer `dp` (frame gap) from the saved flow magnitudes.

In [ ]:
with h5py.File(pix_path, 'r') as f:
    pair_keys = sorted(k for k in f['pairs'].keys() if not k.startswith('_'))
    print(f'n_pairs = {len(pair_keys)}\n')

    print(f'{"pair":>6} | {"speed max":>10} | {"speed mean":>10} | {"|u| max":>9} | {"|v| max":>9}')
    print('-' * 65)
    all_speeds = []
    for k in pair_keys:
        uA = f[f'pairs/{k}/fields/uA'][...]
        vA = f[f'pairs/{k}/fields/vA'][...]
        s = np.hypot(uA, vA)
        all_speeds.append(s)
        print(f'{k:>6} | {s.max():>10.3f} | {s.mean():>10.3f} | {abs(uA).max():>9.3f} | {abs(vA).max():>9.3f}')
    all_speeds = np.stack(all_speeds)
    print(f'\nacross all pairs: speed max = {all_speeds.max():.3f}, mean = {all_speeds.mean():.3f}, std = {all_speeds.std():.3f}')

In [ ]:
# Infer dp (number of saved frames between A and B) from the flow field magnitudes.
# Saved uA = velocity * Δt_pair; Δt_pair = dp * Δt_save. So dp = round(median(uA/u_raw) / Δt_save).
with h5py.File(combined_path, 'r') as f:
    t_keys = sorted([k for k in f['fields/timeseries/t'].keys() if not k.startswith('_')],
                    key=lambda s: int(s) if s.isdigit() else -1)
    t_vals = np.array([float(np.asarray(f[f'fields/timeseries/t/{k}'])) for k in t_keys])
    dt_save = t_vals[1] - t_vals[0]
    u_raw_frame0 = np.asarray(f[f'fields/timeseries/u/{t_keys[0]}']).squeeze()

with h5py.File(pix_path, 'r') as f:
    uA1 = f[f'pairs/000001/fields/uA'][...]

mask = abs(u_raw_frame0) > 0.1
ratio_median = np.median(uA1[mask] / u_raw_frame0[mask])
dp = int(round(ratio_median / dt_save))

print(f'frame iteration keys: {t_keys}')
print(f'sim times:           {np.round(t_vals, 4).tolist()}')
print(f'Δt_save  = {dt_save:.4f} s')
print(f'inferred dp = {dp}   (Δt_pair = {dp * dt_save:.4f} s, {dp}× one saved interval)')

**Reading the table.** Speed max should land near ~10 px/pair (target was 11; achieved ~10 because `dp=2` and `smax ≈ 5` per saved interval). Per-pair stats should be similar across all 4 pairs since the same flow physics is sampled at slightly different times.

## 2. Visual: frames A and B

Load the focus pair, plot A and B side by side. With `dp=2`, A and B are two saved-frames apart in time.

In [ ]:
key = f'{PAIR:06d}'
with h5py.File(pix_path, 'r') as f:
    A  = f[f'pairs/{key}/A'][...]
    B  = f[f'pairs/{key}/B'][...]
    uA = f[f'pairs/{key}/fields/uA'][...]
    vA = f[f'pairs/{key}/fields/vA'][...]

print(f'pair {PAIR} loaded:  A={A.shape}{A.dtype}  uA={uA.shape}{uA.dtype}')
print(f'  A: nonzero={(A>0).sum():,}, total intensity={A.sum():,}, mean={A.mean():.2f}')
print(f'  B: nonzero={(B>0).sum():,}, total intensity={B.sum():,}, mean={B.mean():.2f}')
print(f'  identical?  {np.array_equal(A, B)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(A, cmap='gray', origin='lower')
axes[0].set_title(f'Frame A  (pair {PAIR}, pix11)')
axes[1].imshow(B, cmap='gray', origin='lower')
axes[1].set_title(f'Frame B  (pair {PAIR}, pix11) — ~10 px later')
for ax in axes:
    ax.set_xlabel('x [px]'); ax.set_ylabel('y [px]')
plt.tight_layout(); plt.show()

## 3. Flow field anatomy: u, v, speed + quiver

Units are **pixels per pair** (the saved fields are already scaled by `Δt_pair`).

In [ ]:
vmax = max(abs(uA).max(), abs(vA).max())
speed = np.hypot(uA, vA)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

im0 = axes[0].imshow(uA, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
axes[0].set_title(f'u  (x-velocity, max={abs(uA).max():.2f} px/pair)')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(vA, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
axes[1].set_title(f'v  (y-velocity, max={abs(vA).max():.2f} px/pair)')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

H, W = speed.shape
step = 24
ys, xs = np.mgrid[0:H:step, 0:W:step]
im2 = axes[2].imshow(speed, cmap='viridis', origin='lower')
axes[2].quiver(xs, ys, uA[::step, ::step], vA[::step, ::step], color='white', scale_units='xy', angles='xy', width=0.003)
axes[2].set_title(f'|v| + direction  (max={speed.max():.2f})')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.set_xlabel('x [px]'); ax.set_ylabel('y [px]')
plt.tight_layout(); plt.show()

## 4. Displacement distribution & percentiles

Quantitative breakdown of u, v, and speed across the whole field.

In [ ]:
def percentile_table(arr, name):
    ps = [1, 5, 25, 50, 75, 95, 99]
    vals = np.percentile(arr, ps)
    row = '  '.join(f'p{p:>2}={v:+.2f}' for p, v in zip(ps, vals))
    print(f'{name:>10}:  min={arr.min():+.2f}  max={arr.max():+.2f}  mean={arr.mean():+.2f}  std={arr.std():.2f}')
    print(f'             {row}')

percentile_table(uA.ravel(), 'u')
percentile_table(vA.ravel(), 'v')
percentile_table(speed.ravel(), 'speed')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(uA.ravel(), bins=80, color='steelblue')
axes[0].set_title('u distribution'); axes[0].set_xlabel('px/pair')
axes[1].hist(vA.ravel(), bins=80, color='indianred')
axes[1].set_title('v distribution'); axes[1].set_xlabel('px/pair')
axes[2].hist(speed.ravel(), bins=80, color='seagreen')
axes[2].set_title('speed distribution'); axes[2].set_xlabel('px/pair')
for ax in axes:
    ax.axvline(0, color='k', linewidth=0.5)
plt.tight_layout(); plt.show()

## 5. Patch FFT cross-correlation — measured vs predicted

Tests whether the **rendered images** carry displacement consistent with the saved flow. For 9 patches across the image: cross-correlate A and B patches to recover the dominant local shift, compare to the average flow over the patch.

In [ ]:
def fft_xcorr_peak(a, b):
    """Return (dy, dx) such that particles in `a` moved by (dy, dx) to land in `b`.
    Uses periodic FFT cross-correlation; mean-subtracted."""
    a = a.astype(np.float32) - a.mean()
    b = b.astype(np.float32) - b.mean()
    corr = np.real(np.fft.ifft2(np.conj(np.fft.fft2(a)) * np.fft.fft2(b)))
    corr = np.fft.fftshift(corr)
    H, W = corr.shape
    py, px = np.unravel_index(np.argmax(corr), corr.shape)
    return py - H // 2, px - W // 2

patch = 64
half = patch // 2
H, W = A.shape
anchors = [(cy, cx) for cy in (H // 4, H // 2, 3 * H // 4)
                     for cx in (W // 4, W // 2, 3 * W // 4)]

measured, predicted = [], []
print(f'{"anchor":>14} | {"measured (dy, dx)":>22} | {"predicted (dy, dx)":>24} | residual')
print('-' * 90)
for (cy, cx) in anchors:
    a_patch = A[cy - half:cy + half, cx - half:cx + half]
    b_patch = B[cy - half:cy + half, cx - half:cx + half]
    dy_m, dx_m = fft_xcorr_peak(a_patch, b_patch)
    dy_p = vA[cy - half:cy + half, cx - half:cx + half].mean()
    dx_p = uA[cy - half:cy + half, cx - half:cx + half].mean()
    res = np.hypot(dy_m - dy_p, dx_m - dx_p)
    measured.append((dy_m, dx_m)); predicted.append((dy_p, dx_p))
    print(f'{(cy, cx)!s:>14} | {f"({dy_m:+d}, {dx_m:+d})":>22} | {f"({dy_p:+.2f}, {dx_p:+.2f})":>24} | {res:.2f} px')

measured = np.array(measured, dtype=float)
predicted = np.array(predicted, dtype=float)
rmse_patches = np.sqrt(((measured - predicted) ** 2).mean())
print(f'\nPatch-level RMSE = {rmse_patches:.3f} px')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for i, (name, m, p) in enumerate([('dy', measured[:, 0], predicted[:, 0]),
                                   ('dx', measured[:, 1], predicted[:, 1])]):
    lim = max(abs(m).max(), abs(p).max(), 1) * 1.2
    axes[i].plot([-lim, lim], [-lim, lim], 'k--', linewidth=0.8, label='y = x')
    axes[i].scatter(p, m, s=60, color='steelblue', edgecolor='k')
    axes[i].set_xlabel(f'predicted {name} [px/pair]')
    axes[i].set_ylabel(f'measured {name} [px/pair]')
    axes[i].set_title(f'{name}-displacement: measured vs predicted')
    axes[i].set_xlim(-lim, lim); axes[i].set_ylim(-lim, lim)
    axes[i].set_aspect('equal'); axes[i].grid(alpha=0.3); axes[i].legend()
plt.tight_layout(); plt.show()

## 6. Per-particle displacement (gold standard)

**This is the strongest possible check** because it bypasses image rendering entirely. The combined JLD2 file stores raw particle positions at every saved frame, with stable particle IDs across frames.

For each of the 16,384 particles:
1. Read `(xA, yA)` at frame `keyA` and `(xB, yB)` at frame `keyB` from the combined file.
2. Compute the **actual** displacement: `Δx = xB - xA`, `Δy = yB - yA` (with periodic-domain wrapping).
3. Look up the **predicted** displacement by bilinearly interpolating the saved `uA, vA` flow field at the particle's starting position `(xA, yA)`.
4. Compare. If they agree to within a fraction of a pixel **for every particle**, the pipeline is internally consistent end-to-end.

In [ ]:
def load_particles(h5, frame_key):
    ds = h5[f'particles/timeseries/particles/{frame_key}']
    x = deref(ds.fields('x')[()], h5).astype(np.float64).ravel()
    y = deref(ds.fields('y')[()], h5).astype(np.float64).ravel()
    return x, y

keyA = t_keys[PAIR - 1]
keyB = t_keys[PAIR - 1 + dp]
print(f'pair {PAIR}:  A is iter {keyA}  (t={t_vals[PAIR-1]:.3f}s)   →   B is iter {keyB}  (t={t_vals[PAIR-1+dp]:.3f}s)')

with h5py.File(combined_path, 'r') as f:
    xA_p, yA_p = load_particles(f, keyA)
    xB_p, yB_p = load_particles(f, keyB)

print(f'particles loaded: n={len(xA_p):,}')
print(f'  position range at A:  x in [{xA_p.min():.2f}, {xA_p.max():.2f}], y in [{yA_p.min():.2f}, {yA_p.max():.2f}]')
print(f'  position range at B:  x in [{xB_p.min():.2f}, {xB_p.max():.2f}], y in [{yB_p.min():.2f}, {yB_p.max():.2f}]')

In [ ]:
DOMAIN_L = 512.0  # grid extent (must match 2DTurbulence.jl)

def wrap_diff(d, L):
    """Wrap signed displacement into [-L/2, L/2). Handles particles crossing periodic boundary."""
    return ((d + L / 2) % L) - L / 2

def bilinear_periodic(field, x, y):
    """Bilinear interpolation of a 2D field at (x, y) pixel coords, with periodic wrap."""
    Hf, Wf = field.shape
    x0 = np.floor(x).astype(int); y0 = np.floor(y).astype(int)
    fx = x - np.floor(x);         fy = y - np.floor(y)
    x0 %= Wf; y0 %= Hf
    x1 = (x0 + 1) % Wf; y1 = (y0 + 1) % Hf
    return ((1 - fx) * (1 - fy) * field[y0, x0]
          +      fx  * (1 - fy) * field[y0, x1]
          + (1 - fx) *      fy  * field[y1, x0]
          +      fx  *      fy  * field[y1, x1])

# Actual displacement per particle (with periodic wrap)
dx_actual = wrap_diff(xB_p - xA_p, DOMAIN_L)
dy_actual = wrap_diff(yB_p - yA_p, DOMAIN_L)

# Predicted displacement from saved flow at the particle's starting position
dx_pred = bilinear_periodic(uA, xA_p, yA_p)
dy_pred = bilinear_periodic(vA, xA_p, yA_p)

# Residual = actual - predicted, per particle
res_x = dx_actual - dx_pred
res_y = dy_actual - dy_pred
res_mag = np.hypot(res_x, res_y)
rmse_particles = np.sqrt((res_mag ** 2).mean())

print(f'                     {"actual":>20}  {"predicted":>20}  {"residual":>20}')
print(f'  |dx|  mean       {abs(dx_actual).mean():>15.4f} px  {abs(dx_pred).mean():>17.4f} px  {abs(res_x).mean():>17.4f} px')
print(f'  |dy|  mean       {abs(dy_actual).mean():>15.4f} px  {abs(dy_pred).mean():>17.4f} px  {abs(res_y).mean():>17.4f} px')
print(f'  speed max        {np.hypot(dx_actual, dy_actual).max():>15.4f} px  {np.hypot(dx_pred, dy_pred).max():>17.4f} px')
print()
print(f'  per-particle RMSE = {rmse_particles:.4f} px   (over {len(xA_p):,} particles)')
print(f'  residual percentiles:  p50={np.percentile(res_mag, 50):.4f}  p95={np.percentile(res_mag, 95):.4f}  p99={np.percentile(res_mag, 99):.4f}  max={res_mag.max():.4f} px')

In [ ]:
# Scatter: predicted vs actual per particle (all 16k points). Should fall on y = x.
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, (name, act, pred) in enumerate([('dx (per particle)', dx_actual, dx_pred),
                                        ('dy (per particle)', dy_actual, dy_pred)]):
    lim = max(abs(act).max(), abs(pred).max()) * 1.1
    axes[i].plot([-lim, lim], [-lim, lim], 'k--', linewidth=0.8, label='y = x')
    axes[i].scatter(pred, act, s=1, alpha=0.15, color='steelblue')
    axes[i].set_xlabel(f'predicted {name} [px]')
    axes[i].set_ylabel(f'actual {name} [px]')
    axes[i].set_title(name)
    axes[i].set_xlim(-lim, lim); axes[i].set_ylim(-lim, lim)
    axes[i].set_aspect('equal'); axes[i].grid(alpha=0.3); axes[i].legend()

# Histogram of residual magnitudes
axes[2].hist(res_mag, bins=80, color='seagreen')
axes[2].axvline(rmse_particles, color='red', linestyle='--', label=f'RMSE = {rmse_particles:.3f} px')
axes[2].set_title('per-particle residual |actual − predicted|')
axes[2].set_xlabel('residual magnitude [px]')
axes[2].set_ylabel('particles')
axes[2].legend()
plt.tight_layout(); plt.show()

## 7. Verdict

Summary of all checks.

In [ ]:
checks = [
    ('file has expected pairs',          len(pair_keys) == 4),
    ('A and B not identical',            not np.array_equal(A, B)),
    ('max speed near pix target',        9.0 <= speed.max() <= 12.0),
    ('patch RMSE < 1 px',                rmse_patches < 1.0),
    ('per-particle RMSE < 0.5 px',       rmse_particles < 0.5),
    ('residual p99 < 1 px',              np.percentile(res_mag, 99) < 1.0),
]

print(f'\n{"check":<35} | result')
print('-' * 55)
for desc, ok in checks:
    mark = '✓ PASS' if ok else '✗ FAIL'
    print(f'{desc:<35} | {mark}')

n_pass = sum(1 for _, ok in checks if ok)
print(f'\n{n_pass}/{len(checks)} checks passed.')
if n_pass == len(checks):
    print('\n→ Dataset is internally consistent. Ready for ML training.')
else:
    print('\n→ One or more checks failed. Investigate before training.')